# Spike Shifted Mutations

Interactive Altair chart of shifted mutations and site-level heatmaps.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import multidms

from notebooks._common import load_config, setup_altair, combine_replicate_muts

In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]
output_dir = spike["output_dir"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
times_seen_threshold = spike["times_seen_threshold"]

warnings.simplefilter("ignore")
%matplotlib inline
setup_altair()

In [ ]:
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))
print(f"Loaded {len(models)} models")

## Interactive shift chart

In [ ]:
mc = multidms.ModelCollection(models.drop(columns="replicate"))
chart = mc.mut_param_heatmap(
    query=f"fusionreg == {chosen_lasso_strength}",
    times_seen_threshold=times_seen_threshold,
)
chart.save(f"{output_dir}/interactive_shift_chart.html")
print(f"Saved interactive_shift_chart.html")
chart

## Shifted mutations analysis

In [ ]:
mut_df_replicates = combine_replicate_muts(
    {
        f"{fit.dataset_name}".split("-")[-1]: fit.model
        for fit in models.query(f"fusionreg == {chosen_lasso_strength}").itertuples()
    },
    predicted_func_scores=False,
    how="inner",
    times_seen_threshold=times_seen_threshold,
)

# Get site map
model = models.query(f"fusionreg == {chosen_lasso_strength}").iloc[0].model
site_map = model.data.site_map

mut_df_replicates["sense"] = [
    "stop" if mut == "*" else "nonsynonymous" for mut in mut_df_replicates.muts
]
print(f"Mutations DataFrame: {len(mut_df_replicates)} rows")
mut_df_replicates.head()

## Shift heatmap (zoomed sites)

In [ ]:
# Get domain and site range info from config
domain_dict = spike.get("domain_dict", {})

# Identify sites with largest shifts
shift_cols = [c for c in mut_df_replicates.columns if "avg_shift" in c]
if shift_cols:
    mut_df_replicates["max_abs_shift"] = mut_df_replicates[shift_cols].abs().max(axis=1)
    top_sites = (
        mut_df_replicates.query("sense == 'nonsynonymous'")
        .groupby("sites")["max_abs_shift"]
        .max()
        .nlargest(20)
        .index.tolist()
    )

    sort_order = [
        "R", "K", "H", "D", "E", "Q", "N", "S",
        "T", "Y", "W", "F", "A", "I", "L", "M",
        "V", "G", "P", "C", "-", "*",
    ]

    # Create a simple heatmap of top shifted sites
    for shift_col in shift_cols:
        condition = shift_col.replace("avg_shift_", "")
        plot_df = mut_df_replicates[mut_df_replicates.sites.isin(top_sites)].copy()
        plot_df["muts"] = pd.Categorical(plot_df["muts"], categories=sort_order, ordered=True)
        pivot = plot_df.pivot_table(index="muts", columns="sites", values=shift_col)
        pivot = pivot.reindex(sort_order)

        fig, ax = plt.subplots(figsize=(12, 6))
        vmax = max(abs(pivot.min().min()), abs(pivot.max().max()))
        sns.heatmap(pivot, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, ax=ax)
        ax.set_title(f"Shift heatmap: {condition}")
        plt.tight_layout()

saveas = "shift_by_site_heatmap_zoom"
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()